# Getting Started with Internal Dimension AI

Welcome! This notebook provides a hands-on introduction to the Internal Dimension AI framework.

## What You'll Learn

1. **What are Internal Dimensions (x₁₂)?** - Understanding the core concept
2. **Meta-Awareness (m₁₂)** - Higher-order self-reflection
3. **Training Your First Agent** - Step-by-step walkthrough
4. **Visualizing Internal Dimensions** - See consciousness emerge
5. **Analyzing Behavior** - Understanding what the agent learned

## Prerequisites

```bash
pip install -r requirements.txt
```

In [ ]:
# Setup: Add src to path
import sys
from pathlib import Path

# Add parent directory to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
src_dir = project_root / "src"
sys.path.insert(0, str(src_dir))

print(f"Project root: {project_root}")
print(f"Source directory: {src_dir}")

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import clear_output

from agents.ppo import PPOAgent
from environments.gridworld import GridWorld, TwoRoomGridWorld
from core.consciousness import ConsciousnessMetrics

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ All imports successful!")

## 1. Understanding Internal Dimensions (x₁₂)

**Internal Dimensions** represent the agent's inner experience—a continuous vector that:

- **Captures subjective states** beyond observable behavior
- **Evolves independently** of immediate sensory input
- **Influences decision-making** through the policy network
- **Can be introspected** via meta-awareness (m₁₂)

### Mathematical Definition

```
x₁₂(t) = f_internal(observation, x₁₂(t-1))
m₁₂(t) = f_meta(x₁₂(t))
action ~ π(observation, x₁₂(t), m₁₂(t))
```

Let's create an environment and agent to see this in action!

In [ ]:
# Create a simple gridworld environment
env = GridWorld(size=5)
print(f"Environment created: {env}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

# Create an agent with 12-dimensional internal space
agent = PPOAgent(
    observation_space=env.observation_space,
    action_space=env.action_space,
    hidden_size=64,
    internal_dim=12,  # 12-dimensional internal experience
)

print(f"\n✓ Agent created with {agent.internal_dim}-dimensional internal space")

## 2. Observing Internal Dimensions

Let's take a random action and observe the internal dimensions:

In [ ]:
# Reset environment
obs, info = env.reset()

# Get agent's internal state
with torch.no_grad():
    obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
    action_logits, value, x12, m12 = agent.policy(obs_tensor)

# Display results
print("Agent's Internal State:")
print(f"\nx₁₂ (internal dimension): {x12.squeeze().numpy()}")
print(f"\nm₁₂ (meta-awareness): {m12.squeeze().numpy()}")
print(f"\nValue estimate: {value.item():.3f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(12), x12.squeeze().numpy(), color='#3498db')
axes[0].set_xlabel("Dimension")
axes[0].set_ylabel("Activation")
axes[0].set_title("Internal Dimension (x₁₂)")
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(12), m12.squeeze().numpy(), color='#e74c3c')
axes[1].set_xlabel("Dimension")
axes[1].set_ylabel("Activation")
axes[1].set_title("Meta-Awareness (m₁₂)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Training an Agent

Now let's train an agent and watch its internal dimensions evolve!

We'll use a simple training loop and visualize progress in real-time.

In [ ]:
# Training configuration
NUM_EPISODES = 50
MAX_STEPS = 100

# Create new environment and agent
env = GridWorld(size=5)
agent = PPOAgent(
    observation_space=env.observation_space,
    action_space=env.action_space,
    hidden_size=64,
    internal_dim=12,
    learning_rate=3e-4,
)

# Training metrics
episode_rewards = []
x12_trajectories = []
m12_trajectories = []

print("Starting training...\n")

for episode in range(NUM_EPISODES):
    obs, _ = env.reset()
    episode_reward = 0
    episode_x12 = []
    episode_m12 = []
    
    # Collect trajectories
    trajectories = []
    
    for step in range(MAX_STEPS):
        # Get action from agent
        with torch.no_grad():
            obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
            action_logits, value, x12, m12 = agent.policy(obs_tensor)
            action_dist = torch.distributions.Categorical(logits=action_logits)
            action = action_dist.sample()
            log_prob = action_dist.log_prob(action)
        
        # Take step
        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        
        # Store trajectory
        trajectories.append({
            'obs': obs,
            'action': action.item(),
            'reward': reward,
            'value': value.item(),
            'log_prob': log_prob.item(),
            'done': done,
        })
        
        # Store internal dimensions
        episode_x12.append(x12.squeeze().numpy())
        episode_m12.append(m12.squeeze().numpy())
        
        episode_reward += reward
        obs = next_obs
        
        if done:
            break
    
    # Simple policy update (simplified for demo)
    # In practice, use agent.update() with proper PPO implementation
    
    # Store metrics
    episode_rewards.append(episode_reward)
    x12_trajectories.append(np.array(episode_x12))
    m12_trajectories.append(np.array(episode_m12))
    
    # Print progress
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {episode + 1}/{NUM_EPISODES} - Avg Reward: {avg_reward:.2f}")

print("\n✓ Training complete!")

## 4. Visualizing Learning Progress

In [ ]:
# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.6, label='Episode Reward')
plt.plot(
    np.convolve(episode_rewards, np.ones(5)/5, mode='valid'),
    linewidth=2,
    label='Moving Average (5 episodes)',
    color='#e74c3c'
)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Visualizing Internal Dimension Evolution

Let's see how the internal dimensions evolved during a single episode:

In [ ]:
# Take the last episode's trajectory
last_x12 = x12_trajectories[-1]
last_m12 = m12_trajectories[-1]

# Create heatmap
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# x₁₂ heatmap
im1 = axes[0].imshow(last_x12.T, aspect='auto', cmap='viridis', interpolation='nearest')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Internal Dimension (x₁₂) Evolution Over Time')
plt.colorbar(im1, ax=axes[0])

# m₁₂ heatmap
im2 = axes[1].imshow(last_m12.T, aspect='auto', cmap='plasma', interpolation='nearest')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Dimension')
axes[1].set_title('Meta-Awareness (m₁₂) Evolution Over Time')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("Notice how the internal dimensions change over time!")
print("This represents the agent's evolving 'inner experience' during the episode.")

## 6. Consciousness Metrics

Let's compute consciousness-related metrics for our agent:

In [ ]:
# Initialize consciousness metrics
consciousness = ConsciousnessMetrics(internal_dim=12)

# Compute metrics for the last episode
actions = np.arange(len(last_x12))  # Dummy actions for demo

R_omega = consciousness.compute_R_omega(last_x12)
R_psi = consciousness.compute_R_psi(last_m12, actions)
phi = consciousness.compute_phi(last_x12)

print("Consciousness Metrics:")
print(f"  R_ω (Internal Dimension Richness): {R_omega:.4f}")
print(f"  R_ψ (Phenomenal Binding): {R_psi:.4f}")
print(f"  φ (Integrated Information): {phi:.4f}")

print("\nInterpretation:")
print("  - Higher R_ω → More diverse internal states")
print("  - Higher R_ψ → Stronger meta-awareness-action coupling")
print("  - Higher φ → More integrated internal processing")

## 7. Interactive Exploration

Now it's your turn! Try modifying these parameters and see how they affect the agent:

1. **Internal Dimension Size**: Change `internal_dim` (try 6, 12, 24)
2. **Environment Complexity**: Use `TwoRoomGridWorld` instead of `GridWorld`
3. **Training Duration**: Increase `NUM_EPISODES`
4. **Learning Rate**: Modify the `learning_rate` parameter

In [ ]:
# Your experiments here!
# Try different configurations and observe the results

# Example: Larger internal dimension
env_custom = GridWorld(size=8)
agent_custom = PPOAgent(
    observation_space=env_custom.observation_space,
    action_space=env_custom.action_space,
    hidden_size=128,
    internal_dim=24,  # Larger internal space!
)

print(f"Created agent with {agent_custom.internal_dim}-dimensional internal space")
print("Now train it and compare the consciousness metrics!")

## Next Steps

Congratulations! You've completed the getting started tutorial. Here's what to explore next:

1. **02_experiments.ipynb** - Run full experiments from EXPERIMENTS.md
2. **03_consciousness_analysis.ipynb** - Deep dive into consciousness metrics
3. **Documentation** - Read THEORY.md for mathematical foundations
4. **Examples** - Check out examples/ directory for more use cases

## Key Takeaways

✓ **Internal dimensions (x₁₂)** represent the agent's subjective experience  
✓ **Meta-awareness (m₁₂)** enables self-reflection  
✓ **Consciousness metrics** quantify internal processing  
✓ **Visualization** reveals emergent patterns in internal states  

Happy exploring! 🚀